In [21]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
if os.getcwd().endswith('notebooks'):
    print('here')
    os.chdir(r'..')
    os.chdir(r'..')
    os.chdir(r'chess_engine')
import sys
sys.path.append('../')
# Local imports (adjust paths to match your project)
from chess_engine.src.model.classes.autoencoder.AE_DataLoader import get_dataloaders, FlattenTransform
from chess_engine.src.model.classes.autoencoder.SingleInputAutoEncoder import SingleInputAutoencoder
from chess_engine.src.model.classes.autoencoder.FrozenEncoder import FrozenEncoder
# from chess_engine.src.model.classes.autoencoder.ExtendedAutoencoder import ExtendedAutoencoder
from chess_engine.src.model.classes.autoencoder.model_operator import train_autoencoder

In [22]:
transform = FlattenTransform()
num_epochs=1
lr=1e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader, test_loader, val_loader = get_dataloaders(transform)
autoencoder = SingleInputAutoencoder(input_dim=836, latent_dim=128).to(device)

In [23]:
train_autoencoder(autoencoder, train_loader, val_loader, num_epochs=num_epochs, lr=lr, device=device)


Epoch [1/2] - Train Loss: 0.0160, Val Loss: 0.0120
Epoch [2/2] - Train Loss: 0.0101, Val Loss: 0.0090


In [32]:
ex_autoencoder = ExtendedAutoencoder(autoencoder,latent_dim=64)
train_autoencoder(ex_autoencoder, train_loader, val_loader, num_epochs=num_epochs, lr=lr, device=device)

Epoch [1/1] - Train Loss: 0.0113, Val Loss: 0.0086


In [34]:
ex_autoencoder = ExtendedAutoencoder(ex_autoencoder,latent_dim=32)
train_autoencoder(ex_autoencoder2, train_loader, val_loader, num_epochs=num_epochs, lr=lr, device=device)

Epoch [1/1] - Train Loss: 0.0120, Val Loss: 0.0115


In [31]:
class ExtendedAutoencoder(nn.Module):
    def __init__(self, trained_autoencoder, latent_dim=64):
        super().__init__()
        
        # The frozen encoder
        self.encoder = trained_autoencoder.get_encoder()
        for param in self.encoder.parameters():
            param.requires_grad = False


        self.latent_dim = latent_dim

        
        
        
        self.enc_latent_layers = nn.Sequential(nn.Linear(trained_autoencoder.latent_dim, self.latent_dim ),
                                     nn.ReLU())


        self.dec_latent_layers = nn.Sequential(nn.Linear(self.latent_dim , trained_autoencoder.latent_dim),
                                                nn.ReLU())
        # Reuse the old decoder
        self.decoder = trained_autoencoder.get_decoder()



    def get_encoder(self):
       encoder = nn.Sequential(self.encoder,
                               self.enc_latent_layers)
       return encoder
    

    def get_decoder(self):
       decoder = nn.Sequential(self.dec_latent_layers,
                               self.decoder)
       return decoder

    def encode(self, x):
       return self.encoder(x)

    def decode(self, z):
       return self.decoder(z)
        
    def forward(self, x):
        # Pass through frozen encoder
        with torch.no_grad():
            z = self.encoder(x)
        z = self.enc_latent_layers(z)
        
        # Pass into the old decoder
        z = self.dec_latent_layers(z)
        x_recon = self.decoder(z)
        return x_recon


In [27]:
print(autoencoder)

SingleInputAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=836, out_features=512, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=256, out_features=128, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=512, out_features=836, bias=True)
  )
)


In [29]:
print(ex_autoencoder)

ExtendedAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=836, out_features=512, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=256, out_features=128, bias=True)
  )
  (enc_latent_layers): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
  )
  (dec_latent_layers): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=512, out_features=836, bias=True)
  )
)


In [30]:
print(ex_autoencoder2)

ExtendedAutoencoder(
  (encoder): Sequential(
    (0): Sequential(
      (0): Linear(in_features=836, out_features=512, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=512, out_features=256, bias=True)
      (3): ReLU(inplace=True)
      (4): Linear(in_features=256, out_features=128, bias=True)
    )
    (1): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
    )
  )
  (enc_latent_layers): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
  )
  (dec_latent_layers): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU(inplace=True)
    (4): Linear(in_features=512, out_features=836, bias=True)
  )
)
